# K-Means Time Series Analysis on English Romance Films

Here I want to analyze the variation of color throughout the length of films. Basically as before splitting a film into 5 segments, applying k_means to that segments, getting 15 ish colors, but unlike before i won't group all the colors together, rather ill analyze each segment separately.

In [27]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML 
import matplotlib.colors as mcolors
import math
from collections import Counter
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans

### Load the data and preprocess

In [28]:
# Path to the directory containing the CSV files
directory_path = '/Users/rsudhir/Documents/GitHub/Data-Science-Project---Colors-Of-Romance/English-Analysis/English-Movie-CSVs'

# List to hold each DataFrame
dfs = []

# Loop through the files in the directory and load each CSV
for i in range(1, 26):
    file_path = os.path.join(directory_path, f'{i}.csv')
    df = pd.read_csv(file_path)
    # Drop columns with NaN values
    df = df.drop(columns=['color_10_r', 'color_10_g', 'color_10_b'])
    
    # Add a column to indicate which movie the data is from
    df['movie_id'] = i
    
    dfs.append(df)

# Combine all DataFrames into a single DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

# Display the combined DataFrame to ensure it looks correct
display(combined_df.head())

,frame_path,color_1_r,color_1_g,color_1_b,color_2_r,color_2_g,color_2_b,color_3_r,color_3_g,color_3_b,...,color_7_r,color_7_g,color_7_b,color_8_r,color_8_g,color_8_b,color_9_r,color_9_g,color_9_b,movie_id
0,output_0000001.png,94,76,59,210,190,185,28,24,21,...,58,26,24,169,174,174,163,160,165,1
1,output_0000002.png,38,36,28,191,177,178,129,114,101,...,164,132,118,156,148,156,100,88,100,1
2,output_0000003.png,126,109,90,28,24,19,196,180,163,...,170,168,165,178,180,163,71,63,34,1
3,output_0000004.png,41,32,30,94,81,72,105,100,90,...,155,156,154,83,89,101,152,148,156,1
4,output_0000005.png,32,27,26,196,174,161,121,93,80,...,119,130,150,148,148,140,73,77,96,1


In [29]:
# Define the number of segments
num_segments = 5

# Loop through each movie and create a segment ID
combined_df['segment_id'] = combined_df.groupby('movie_id').cumcount() // (combined_df.groupby('movie_id')['frame_path'].transform('count') // num_segments)

### K_means on each segment

For each segment of each movie i am getting x colors

In [30]:
# Apply K-Means Clustering to Each Segment Separately
num_clusters_per_segment = 15  # Number of clusters per segment

# Create a list to hold cluster centers for each segment
segment_clusters = []

# Group by movie and segment
grouped = combined_df.groupby(['movie_id', 'segment_id'])

for (movie_id, segment_id), group in grouped:
    # Extract RGB values for the dominant colors in this segment
    colors = []
    for i in range(1, 10):
        colors.append(group[[f'color_{i}_r', f'color_{i}_g', f'color_{i}_b']].dropna().values)
    
    # Combine colors into a single array
    colors_array = np.vstack(colors)

    # Ensure sufficient data points for clustering
    if len(colors_array) >= num_clusters_per_segment:
        # Perform K-Means clustering
        kmeans = KMeans(n_clusters=num_clusters_per_segment, n_init='auto', random_state=42)
        kmeans.fit(colors_array)
        
        # Store the cluster centers for this segment
        segment_clusters.append((movie_id, segment_id, kmeans.cluster_centers_))
    else:
        print(f"Skipping segment {segment_id} of movie {movie_id} due to insufficient data points.")


Skipping segment 5 of movie 6 due to insufficient data points.
Skipping segment 5 of movie 7 due to insufficient data points.
Skipping segment 5 of movie 8 due to insufficient data points.
Skipping segment 5 of movie 19 due to insufficient data points.


### Visualization of all segments and their colors

Have commented out the code as its quite a long output, feel free to uncomment if you'd like to see all the colors.

In [31]:
# # Visualization and Analysis
# for movie_id, segment_id, centers in segment_clusters:
#     # Convert RGB values to HEX
#     hex_colors = [mcolors.to_hex([r/255, g/255, b/255]) for r, g, b in centers]

#     # Sort colors by hue for better visualization
#     hsv_colors = [mcolors.rgb_to_hsv([r/255, g/255, b/255]) for r, g, b in centers]
#     sorted_indices = sorted(range(len(hsv_colors)), key=lambda i: (hsv_colors[i][0], hsv_colors[i][1], hsv_colors[i][2]))
#     sorted_hex_colors = [hex_colors[i] for i in sorted_indices]

#     # Create an HTML table for the colors
#     num_columns = 5  # Adjust as needed
#     num_rows = math.ceil(len(sorted_hex_colors) / num_columns)
#     html_table = '<table style="border-collapse: collapse;">'
#     for i in range(num_rows):
#         html_table += '<tr>'
#         for j in range(num_columns):
#             index = i * num_columns + j
#             if index < len(sorted_hex_colors):
#                 hex_color = sorted_hex_colors[index]
#                 html_table += f'<td style="background-color:{hex_color}; width:50px; height:25px; border: 1px solid #ccc;"></td>'
#                 html_table += f'<td style="padding: 5px;">{hex_color}</td>'
#         html_table += '</tr>'
#     html_table += '</table>'
    
#     # Display the HTML table with the colors for this segment
#     display(HTML(f"<h3>Movie {movie_id} - Segment {segment_id}</h3>"))
#     display(HTML(html_table))


### K_means Across Grouped Segments

In [32]:
# Group by segment_id across all movies
combined_segments = {}

for segment_id in range(num_segments):
    segment_colors = []
    for movie_id, _, centers in segment_clusters:
        if segment_id == _:
            segment_colors.append(centers)
    
    # Combine all colors from this segment across movies into a single array
    combined_segments[segment_id] = np.vstack(segment_colors)

In [33]:
# Define the number of clusters for each combined segment
num_clusters_per_combined_segment = 30  # Adjust this as needed

# Store the final cluster centers for each combined segment
final_segment_clusters = {}

for segment_id, colors in combined_segments.items():
    if len(colors) >= num_clusters_per_combined_segment:
        # Perform K-Means clustering on the combined colors for this segment
        kmeans = KMeans(n_clusters=num_clusters_per_combined_segment, n_init='auto', random_state=42)
        kmeans.fit(colors)
        
        # Store the cluster centers for this segment
        final_segment_clusters[segment_id] = kmeans.cluster_centers_
    else:
        print(f"Skipping combined segment {segment_id} due to insufficient data points.")

### Visualizing Colors Across Segments

In [34]:
# Visualize the colors for each combined segment
for segment_id, centers in final_segment_clusters.items():
    # Convert RGB values to HEX
    hex_colors = [mcolors.to_hex([r/255, g/255, b/255]) for r, g, b in centers]

    # Sort colors by hue for better visualization
    hsv_colors = [mcolors.rgb_to_hsv([r/255, g/255, b/255]) for r, g, b in centers]
    sorted_indices = sorted(range(len(hsv_colors)), key=lambda i: (hsv_colors[i][0], hsv_colors[i][1], hsv_colors[i][2]))
    sorted_hex_colors = [hex_colors[i] for i in sorted_indices]

    # Create an HTML table for the colors
    num_columns = 5  # Adjust as needed
    num_rows = math.ceil(len(sorted_hex_colors) / num_columns)
    html_table = '<table style="border-collapse: collapse;">'
    for i in range(num_rows):
        html_table += '<tr>'
        for j in range(num_columns):
            index = i * num_columns + j
            if index < len(sorted_hex_colors):
                hex_color = sorted_hex_colors[index]
                html_table += f'<td style="background-color:{hex_color}; width:50px; height:25px; border: 1px solid #ccc;"></td>'
                html_table += f'<td style="padding: 5px;">{hex_color}</td>'
        html_table += '</tr>'
    html_table += '</table>'
    
    # Display the HTML table with the colors for this segment
    display(HTML(f"<h3>Combined Segment {segment_id}</h3>"))
    display(HTML(html_table))

,#a3574b,,#8e4033,,#aa542e,,#4e3729,,#2b2520
,#a38162,,#725f4d,,#817261,,#171512,,#594d3b
,#b5a183,,#d3c7b4,,#bab3a1,,#878378,,#919591
,#469d48,,#c0c8c2,,#a5aba7,,#333835,,#585d5c
,#cedddb,,#2fbba9,,#196671,,#677478,,#92bfd0
,#2882a8,,#41494d,,#718fa1,,#455b80,,#f5f4f7


,#ac5948,,#9f4531,,#9d6e5a,,#543624,,#977d6a
,#654b38,,#79614c,,#41362d,,#1c1813,,#30281e
,#b8a48a,,#b49761,,#c7c0ae,,#9b937b,,#4d4b42
,#7d7c78,,#6b6b64,,#a7a8a1,,#d5d6cb,,#7e865a
,#949c97,,#348f87,,#545c5f,,#cedae1,,#a8b1b6
,#81adca,,#67889f,,#38454e,,#848b91,,#516c92


,#a64e43,,#854b2e,,#ac632c,,#543c2a,,#30261e
,#866a50,,#a07955,,#5f5143,,#191611,,#c1a477
,#6f685c,,#3c372f,,#9f8f72,,#827d71,,#bab3a2
,#cbc7bb,,#a3a08e,,#dad6b8,,#6cb82d,,#868e8e
,#378d93,,#d3dadb,,#b2c0c2,,#57abc6,,#566064
,#3d484f,,#97a0a6,,#687b8a,,#7da0c5,,#a36f70


,#771e10,,#a65d4a,,#9d4a28,,#7f604b,,#4d392b
,#332921,,#614b37,,#171310,,#cd9b5a,,#97856d
,#bdae98,,#ad9c81,,#85775e,,#cac5ab,,#f0f4e6
,#61655d,,#d8dcd7,,#c7cec5,,#4b4e4c,,#8b908c
,#9ca4a2,,#727a7a,,#aabcc1,,#3a707f,,#28424f
,#77a4c7,,#6582a0,,#4d55d0,,#1a1286,,#d12026


,#a74c40,,#a0624c,,#954a28,,#573d2b,,#1e1a16
,#ad8c69,,#322b24,,#82725e,,#645c51,,#bdbab5
,#534b3f,,#c9b390,,#a29b90,,#141210,,#8f8879
,#f2c632,,#d1d1c4,,#66a816,,#e0e8e3,,#3e4140
,#9fa8a8,,#8a9494,,#656e6f,,#afd4de,,#90b1c0
,#2c4c5f,,#718089,,#51595f,,#4e7b9e,,#484dd3
